In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [3]:
os.chdir("/home/lishaogang/NEU2026-Jan")

In [ ]:
adata_Myeloid = sc.read_h5ad('adata_Myeloid_raw.h5ad')

In [ ]:
# 2. 重新过滤基因（只保留在当前 T 细胞子集中有表达的基因）
sc.pp.filter_genes(adata_Myeloid, min_cells=3)

# 3. 运行 HVG
print("\n--- Finding HVGs within the T-cell subset ---")
# 方案 A: 依然使用 batch_key（如果样本量大且细胞多，推荐）
try:
    sc.pp.highly_variable_genes(adata_Myeloid, batch_key='dataset', n_top_genes=2000, flavor='seurat')
except Exception as e:
    print(f"依然报错: {e}。自动切换到方案 B：全局 HVG 计算...")
    # 方案 B: 如果还是报错，说明样本间差异过大或仍有极小样本，则不带 batch_key 运行
    sc.pp.highly_variable_genes(adata_Myeloid, n_top_genes=2000, flavor='seurat')

# 4. 提取 HVG 并后续处理
adata_Myeloid_hvg = adata_Myeloid[:, adata_Myeloid.var.highly_variable].copy()
sc.pp.scale(adata_Myeloid_hvg, zero_center=False)
sc.tl.pca(adata_Myeloid_hvg, n_comps=50, random_state=seed)

print("✅ PCA 计算完成！")

In [ ]:
# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size before plotting
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_Myeloid_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [ ]:

print("\n--- Re-running Harmony on the T-cell subset ---")
sc.external.pp.harmony_integrate(
    adata_Myeloid_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

adata_Myeloid.obsm['X_pca'] = adata_Myeloid_hvg.obsm['X_pca']
adata_Myeloid.obsm['X_pca_harmony'] = adata_Myeloid_hvg.obsm['X_pca_harmony']

In [ ]:
# 计算邻居和 UMAP (建议将邻居数调小一点，比如 10-15，因为 NK 子集通常比较小)
sc.pp.neighbors(adata_Myeloid, n_neighbors=15,n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_Myeloid)


In [ ]:
sc.tl.leiden(adata_Myeloid, resolution=1.0, key_added='leiden_Myeloid')

# --- 7. 可视化 ---


In [ ]:
sc.pl.umap(adata_Myeloid, color=['leiden_Myeloid', 'dataset', 'Disease'], wspace=0.4)

In [ ]:
gene_list_main = [
    'CD3D', 'CD3E', 'CD3G',# T cells
    'NKG7',  'FGFBP2', # NK
    'KLRB1','TRAC','TRBC1','TRBC2', # NKT
    'MS4A1', 'CD79A', 'CD19', # B cell
    'IGHG1', 'MZB1',  # Plasma
    'LILRA4', 'IL3RA', 'CLEC4C', # pDC
    'CD8A', 'CD4', # T cell
    'TRGV9', 'TRDV2', # Gamma-Delta T cell
    #'PPBP','HBB'
    ]# Platelet and Erythrocyte


gene_list_myeloid = [
    'LYZ', 'S100A8', 'S100A9', 'VCAN', # Myeloid
    # cDC1 markers
    'XCR1',
    # cDC2 markers
    'CD1C','CCL22', 'CLEC10A','FCER1A',
    #Monocyte markers
    'CD14', 'FCGR3A','FCN1', 'S100A4', 'S100B','CD163',
    #Macrophage markers
    'C1QA', 'C1QC',
    #Microglia markers
     'P2RY12',
    #Mast cell markers
    'KIT','S100B',
    #ASDC
    'AXL','SIGLEC1'
]

In [ ]:
mac_subset_marker_genes = [
    'APOE', 'C1QA', 'C1QC', 'CD3D', 'CD68', 'CXCL8', 'EPCAM', 'IL1B',
    'LYZ', 'MS4A1', 'PPBP', 'SQLE', 'SPP1', 'TREM2','C1QA','C1QC'
]
genes_to_plot = list(set(gene_list_main + gene_list_myeloid + mac_subset_marker_genes))  

In [ ]:
sc.tl.leiden(adata_Myeloid, resolution=1.0, key_added='leiden_Myeloid', random_state=seed)
sc.pl.umap(adata_Myeloid, color='leiden_Myeloid', legend_loc='on data', title='Myeloid Sub-clusters')

sc.pl.umap(adata_Myeloid, color=gene_list_main, legend_loc='on data')
sc.pl.umap(adata_Myeloid, color=gene_list_myeloid, legend_loc='on data')



In [ ]:
genes_to_plot =  [
    # T细胞
    'CD3D', 'CD3E', 'CD3G', 'TRAC', 'TRBC1', 'TRBC2', 'TRDV2', 'TRGV9', 'CD4', 'CD8A', 'KLRB1', 'NKG7',
    # B细胞/浆细胞
    'CD19', 'CD79A', 'MS4A1', 'IGHG1', 'MZB1',
    # 单核/巨噬细胞
    'CD14', 'CD68', 'CD163', 'C1QA', 'C1QC', 'APOE', 'TREM2', 'SPP1', 'AXL', 'SIGLEC1', 'LYZ', 'FCN1', 'VCAN', 'FGFBP2',
    # 中性粒细胞/粒细胞
    'S100A8', 'S100A9', 'CXCL8', 'IL1B', 'PPBP',
    # DC细胞
    'CD1C', 'CLEC10A', 'CLEC4C', 'LILRA4', 'XCR1',
    # 嗜碱性/肥大细胞
    'FCER1A', 'IL3RA',
    # 其他
    'EPCAM', 'CCL22', 'S100A4', 'S100B', 'SQLE', 'KIT','FCGR3A'
]

In [ ]:
# --- Plot Dot Plot for Cell Type Annotation ---
# Dot plot is an excellent way to show the expression of specific genes in each cluster.
# The size of the dot represents the proportion of cells expressing the gene, and the color intensity represents the average expression level.
# **`use_raw=True`** is a key parameter that ensures we use the normalized and log-transformed raw expression values,
# rather than scaled values, which is more accurate for biological interpretation.
print("Generating dot plot...")
dotplot = sc.pl.dotplot(
    adata_Myeloid,
    genes_to_plot,
    groupby="leiden_Myeloid",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(genes_to_plot)) - 0.5, xmin=-0.5, xmax=len(adata_Myeloid.obs['leiden_Myeloid'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_Myeloid.obs['leiden_Myeloid'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(genes_to_plot) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


print("\n--- Exporting Raw Data of Dot Plot ---")
mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print("\n--- Mean Expression Matrix (Color) ---")
#     #print(mean_expression_df)
#     print("\n--- Cell Expression Fraction Matrix (Dot Size) ---")
#     #print(fraction_df)



# 2. Call the function to generate a summary table
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

# 3. Convert the resulting DataFrame to a tab-separated string and print it
tsv_output = final_summary_df.to_csv(sep='\t')

#print("--- Gene Expression Level Summary Table (Tab-Separated) ---")
print(tsv_output)

In [ ]:

summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)

In [ ]:
# --- Perform Annotation Using apply Method (More Concise Than for Loop) ---
annotations_list = final_summary_df.apply(anno_self.annotate_csf_myeloid, axis=1)

# --- Create Final Concise Report ---
results_df = pd.DataFrame({
    'Cluster': final_summary_df.index,
    'Annotation': annotations_list
})


In [ ]:

# --- Generate Explanation ---
results_df['Explanation'] = results_df.apply(
    anno_self.generate_myeloid_explanation, 
    axis=1, 
    original_df=final_summary_df
)

# --- Print Final Report ---
print("\n\n--- Final Annotation Report ---")
anno_self.pretty_print_df(results_df)


In [ ]:
annotation_dict = results_df.set_index('Cluster')['Annotation'].to_dict()

In [ ]:
annotation_dict = {'0': 'Classical Monocytes',
 '1': 'Classical Monocytes',
 '2': 'cDC2',
 '3': 'Macrophage',
 '4': 'Macrophage',
 '5': 'Classical Monocytes',
 '6': 'Non-classical Monocytes',
 '7': 'Classical Monocytes',
 '8': 'Classical Monocytes',
 '9': 'cDC2',
 '10': 'Lipid-Associated Macrophage',
 '11': 'Macrophage',
 '12': 'Classical Monocytes',
 '13': 'ASDC',
 '14': 'Classical Monocytes',
 '15': 'B_Doublet',
 '16': 'Macrophage',
 '17': 'Classical Monocytes',
 '18': 'Neutrophils'}

In [ ]:
# Apply mapping to create a new 'cell_subtype' column
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs['leiden_Myeloid'].map(annotation_dict).astype('category')

# View the count of roughly annotated cells
print("--- Broad Annotation Counts ---")
print(adata_Myeloid.obs['cell_subtype'].value_counts())

print("\n" + "="*30 + "\n")  # Separator



# Visualize the broad annotations


In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
# 1. 进行注释映射 (确保 annotation_dict 已经定义好)
# 这里的 .obs['leiden_Myeloid'] 必须和你的字典键对应
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs['leiden_Myeloid'].map(annotation_dict).astype('category')

# 2. 查看注释后的细胞数量
print("--- Broad Annotation Counts ---")
print(adata_Myeloid.obs['cell_subtype'].value_counts())

print("\n" + "="*30 + "\n")  # 分隔符

# 3. 可视化注释结果
# 注意：这里请统一使用 adata_Myeloid，并修正 color 拼写
sc.pl.umap(adata_Myeloid, color='cell_subtype', frameon=False, title='Myeloid Cell Subtypes')

In [ ]:
# 1. 剔除第 6 簇
print(f"剔除前细胞数: {adata_Myeloid.n_obs}")
adata_Myeloid = adata_Myeloid[adata_Myeloid.obs['leiden_Myeloid'] != '15', :].copy()
print(f"剔除后细胞数: {adata_Myeloid.n_obs}")


In [ ]:

# 2. 强制对齐旧的 Harmony 坐标 (防止报错)
# 这一步是为了不重跑 Harmony 也能直接画图
adata_Myeloid.obsm['X_pca_harmony'] = adata_Myeloid_hvg[adata_Myeloid.obs_names, :].obsm['X_pca_harmony']

# 3. 重新计算 UMAP（删除了污染簇后，UMAP 布局会更清晰）
sc.pp.neighbors(adata_Myeloid, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_Myeloid)
sc.tl.leiden(adata_Myeloid, resolution=1.0, key_added='leiden_Myeloid')
# 4. 可视化查看清理后的结果
sc.pl.umap(adata_Myeloid, color=['leiden_Myeloid', 'dataset'], title='Myeloid-cell Subset (Cleaned)')

In [ ]:
sc.pl.umap(adata_Myeloid, color='cell_subtype')

In [ ]:
# --- Plot Dot Plot for Cell Type Annotation ---
# Dot plot is an excellent way to show the expression of specific genes in each cluster.
# The size of the dot represents the proportion of cells expressing the gene, and the color intensity represents the average expression level.
# **`use_raw=True`** is a key parameter that ensures we use the normalized and log-transformed raw expression values,
# rather than scaled values, which is more accurate for biological interpretation.
print("Generating dot plot...")
dotplot = sc.pl.dotplot(
    adata_Myeloid,
    genes_to_plot,
    groupby="leiden_Myeloid",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(genes_to_plot)) - 0.5, xmin=-0.5, xmax=len(adata_Myeloid.obs['leiden_Myeloid'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_Myeloid.obs['leiden_Myeloid'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(genes_to_plot) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


print("\n--- Exporting Raw Data of Dot Plot ---")
mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print("\n--- Mean Expression Matrix (Color) ---")
#     #print(mean_expression_df)
#     print("\n--- Cell Expression Fraction Matrix (Dot Size) ---")
#     #print(fraction_df)



# 2. Call the function to generate a summary table
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

# 3. Convert the resulting DataFrame to a tab-separated string and print it
tsv_output = final_summary_df.to_csv(sep='\t')

#print("--- Gene Expression Level Summary Table (Tab-Separated) ---")
print(tsv_output)

In [ ]:
# --- Perform Annotation Using apply Method (More Concise Than for Loop) ---
annotations_list = final_summary_df.apply(anno_self.annotate_csf_myeloid, axis=1)

# --- Create Final Concise Report ---
results_df = pd.DataFrame({
    'Cluster': final_summary_df.index,
    'Annotation': annotations_list
})


In [ ]:

# --- Generate Explanation ---
results_df['Explanation'] = results_df.apply(
    anno_self.generate_myeloid_explanation, 
    axis=1, 
    original_df=final_summary_df
)

# --- Print Final Report ---
print("\n\n--- Final Annotation Report ---")
anno_self.pretty_print_df(results_df)


In [ ]:
annotation_dict = results_df.set_index('Cluster')['Annotation'].to_dict()
annotation_dict

In [ ]:
annotation_dict = {'0': 'Classical Monocytes',
 '1': 'cDC2',
 '2': 'Classical Monocytes',
 '3': 'Macrophage',
 '4': 'Classical Monocytes',
 '5': 'Macrophage',
 '6': 'Non-classical Monocytes',
 '7': 'Classical Monocytes',
 '8': 'Classical Monocytes',
 '9': 'cDC2',
 '10': 'Macrophage',
 '11': 'Macrophage',
 '12': 'Classical Monocytes',
 '13': 'ASDC',
 '14': 'Macrophage',
 '15': 'Classical Monocytes',
 '16': 'Macrophage',
 '17': 'Neutrophils'}



In [ ]:
# Apply mapping to create a new 'cell_subtype' column
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs['leiden_Myeloid'].map(annotation_dict).astype('category')

# View the count of roughly annotated cells
print("--- Broad Annotation Counts ---")
print(adata_Myeloid.obs['cell_subtype'].value_counts())

print("\n" + "="*30 + "\n")  # Separator



# Visualize the broad annotations


In [ ]:
# 1. 进行注释映射 (确保 annotation_dict 已经定义好)
# 这里的 .obs['leiden_Myeloid'] 必须和你的字典键对应
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs['leiden_Myeloid'].map(annotation_dict).astype('category')

# 2. 查看注释后的细胞数量
print("--- Broad Annotation Counts ---")
print(adata_Myeloid.obs['cell_subtype'].value_counts())

print("\n" + "="*30 + "\n")  # 分隔符

# 3. 可视化注释结果
# 注意：这里请统一使用 adata_Myeloid，并修正 color 拼写
sc.pl.umap(adata_Myeloid, color='cell_subtype', frameon=False, title='Myeloid Cell Subtypes')

In [ ]:
sc.pl.umap(adata_Myeloid, color='leiden_Myeloid', frameon=False, title='Myeloid Cell Subtypes',legend_loc = 'on data')

In [ ]:
#保存adata_Myeloid
adata_Myeloid.write_h5ad("adata_Myeloid_final0.h5ad")

In [ ]:
# 筛选出细胞亚型为Macrophage的巨噬细胞数据
adata_Mac = adata_Myeloid[adata_Myeloid.obs['cell_subtype']=='Macrophage'].copy()

# 方法1：最直接的方式 - 通过adata的形状获取细胞数（行数）
cell_count = adata_Mac.shape[0]
print(f"Macrophage细胞数量: {cell_count}")


In [ ]:
sc.pl.umap(adata_Mac, color='leiden_Myeloid', frameon=False, title='Myeloid Cell Subtypes')

In [ ]:
# 2. 强制对齐旧的 Harmony 坐标 (防止报错)
# 这一步是为了不重跑 Harmony 也能直接画图
adata_Mac.obsm['X_pca_harmony'] = adata_Myeloid_hvg[adata_Mac.obs_names, :].obsm['X_pca_harmony']

# 3. 重新计算 UMAP（删除了污染簇后，UMAP 布局会更清晰）
sc.pp.neighbors(adata_Mac, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.umap(adata_Mac)


In [ ]:
sc.tl.leiden(adata_Mac, resolution=0.3, key_added='leiden_Mac')
# 4. 可视化查看清理后的结果
sc.pl.umap(adata_Mac, color=['leiden_Mac', 'dataset'], title='Mac -cell Subset (Cleaned)')

In [165]:
# 计算每个簇与其他所有簇相比的上调差异基因
# 使用wilcoxon检验，只保留上调基因
marker_genes = sc.tl.rank_genes_groups(
    adata_Mac, 
    groupby='leiden_Mac', 
    method='wilcoxon', 
    use_raw=False,
    corr_method='benjamini-hochberg',
    key_added='rank_genes_leiden',
    only_positive=True  # 只保留上调基因
)

# 提取每个簇的top10上调基因
top10_genes = {}
for cluster in adata_Mac.obs['leiden_Mac'].cat.categories:
    # 获取该簇的差异基因列表
    cluster_genes = adata_Mac.uns['rank_genes_leiden']['names'][cluster]
    # 取前10个
    top10_genes[cluster] = cluster_genes[:10].tolist()

# 将所有top10基因合并成一个列表，用于后续绘图
all_top10 = list(set([gene for genes in top10_genes.values() for gene in genes]))

In [166]:
# 绘制dotplot
sc.pl.dotplot(
    adata_Mac, 
    var_names=top10_genes,  # 使用按簇整理的基因列表
    groupby='leiden_Mac',
    title='Macrophage Cluster Marker Genes (Top10 Upregulated)',
    figsize=(15, 8),
    dot_max=0.7,
    standard_scale='var' # 对基因表达进行标准化
)

In [167]:
# 计算每个簇与其他所有簇相比的上调差异基因
# 使用wilcoxon检验，只保留上调基因
marker_genes = sc.tl.rank_genes_groups(
    adata_Mac, 
    groupby='leiden_Mac', 
    method='wilcoxon', 
    use_raw=False,
    corr_method='benjamini-hochberg',
    key_added='rank_genes_leiden',
    only_positive=True  # 只保留上调基因
)

# 提取每个簇的top5上调基因
top5_genes = {}
for cluster in adata_Mac.obs['leiden_Mac'].cat.categories:
    # 获取该簇的差异基因列表
    cluster_genes = adata_Mac.uns['rank_genes_leiden']['names'][cluster]
    # 取前5个
    top5_genes[cluster] = cluster_genes[:5].tolist()

# 将所有top5基因合并成一个列表，用于后续绘图
all_top5 = list(set([gene for genes in top5_genes.values() for gene in genes]))


# 绘制dotplot
sc.pl.dotplot(
    adata_Mac, 
    var_names=top5_genes,  # 使用按簇整理的基因列表
    groupby='leiden_Mac',
    title='Macrophage Cluster Marker Genes (Top10 Upregulated)',
    figsize=(10, 8),
    dot_max=0.7,
    standard_scale='var' # 对基因表达进行标准化
)

In [168]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ========================
# 1. 重新提取簇2的上调差异基因（严格筛选）
# ========================
target_cluster = '2'

# 从rank_genes_groups结果中提取关键信息：基因名、log2FC、p值
rank_res = adata_Mac.uns['rank_genes_leiden']
# 提取簇2的差异基因信息
cluster2_res = {
    'names': rank_res['names'][target_cluster],
    'logfoldchanges': rank_res['logfoldchanges'][target_cluster],
    'pvals_adj': rank_res['pvals_adj'][target_cluster]
}
# 转为DataFrame，方便筛选
cluster2_df = pd.DataFrame(cluster2_res)

# 严格筛选：仅保留 上调（log2FC>0）+ 显著（pvals_adj<0.05） 的基因
# log2FC>0.5 可根据需求调整，值越大筛选越严格
cluster2_up_genes = cluster2_df[
    (cluster2_df['logfoldchanges'] > 0.5) & 
    (cluster2_df['pvals_adj'] < 0.05)
]['names'].tolist()

# 取top10真正上调的基因（若不足10则取全部）
top10_up_genes = cluster2_up_genes[:10]
print(f"簇2真正上调的top10基因：{top10_up_genes}")

# ========================
# 2. 绘制仅上调基因的dotplot（对比簇2 vs Others）
# ========================
# 确保已创建cluster2_vs_others分组（若未创建）
if 'cluster2_vs_others' not in adata_Mac.obs.columns:
    adata_Mac.obs['cluster2_vs_others'] = adata_Mac.obs['leiden_Mac'].apply(
        lambda x: f'Cluster {target_cluster}' if x == target_cluster else 'Others'
    )

# 绘制dotplot（只展示上调基因）
sc.pl.dotplot(
    adata_Mac,
    var_names=top10_up_genes,  # 仅用严格筛选的上调基因
    groupby='cluster2_vs_others',
    title=f'Cluster {target_cluster} Top10 Truly Upregulated Marker Genes',
    figsize=(12, 4),
    dot_max=0.7,
    standard_scale='var',
    show=True
)

# 保存图片
plt.tight_layout()
plt.savefig(f'cluster{target_cluster}_true_upregulated_dotplot.png', dpi=300, bbox_inches='tight')

In [161]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ========================
# 1. 重新计算所有簇的上调差异基因（确保only_positive=True）
# ========================
sc.tl.rank_genes_groups(
    adata_Mac, 
    groupby='leiden_Myeloid', 
    method='wilcoxon', 
    use_raw=False,
    corr_method='benjamini-hochberg',
    key_added='rank_genes_leiden_up',
    only_positive=True
)

# ========================
# 2. 提取每个簇的top10真正上调基因（严格筛选）
# ========================
LOG2FC_THRESHOLD = 0.5
PVAL_ADJ_THRESHOLD = 0.05
cluster_top10_up = {}
all_clusters = adata_Mac.obs['leiden_Myeloid'].cat.categories.tolist()

for cluster in all_clusters:
    # 提取差异基因结果并筛选
    rank_res = adata_Mac.uns['rank_genes_leiden_up']
    cluster_res = pd.DataFrame({
        'names': rank_res['names'][cluster],
        'logfoldchanges': rank_res['logfoldchanges'][cluster],
        'pvals_adj': rank_res['pvals_adj'][cluster]
    })
    # 仅保留显著上调基因
    cluster_up_genes = cluster_res[
        (cluster_res['logfoldchanges'] > LOG2FC_THRESHOLD) &
        (cluster_res['pvals_adj'] < PVAL_ADJ_THRESHOLD)
    ]['names'].tolist()
    cluster_top10_up[cluster] = cluster_up_genes[:10]
    print(f"簇{cluster}上调top10基因：{cluster_top10_up[cluster]}")

# ========================
# 3. 绘制dotplot（完全兼容版，无参数冲突）
# ========================
# 方案1：按簇展示各自top10基因（核心需求）
# 先合并所有簇的top10基因，避免字典传参问题
all_genes = []
gene_cluster_map = {}  # 记录基因所属簇，用于分组显示
for cluster in all_clusters:
    genes = cluster_top10_up[cluster]
    all_genes.extend(genes)
    for gene in genes:
        gene_cluster_map[gene] = f'Cluster {cluster}'

# 绘制dotplot（仅传基因列表，避免字典导致的底层错误）
sc.pl.dotplot(
    adata_Mac,
    var_names=all_genes,  # 传基因列表而非字典，兼容所有版本
    groupby='leiden_Myeloid',
    title='Macrophage Clusters - Top10 Upregulated Marker Genes',
    figsize=(20, 10),  # 加宽画布避免基因名重叠
    dot_max=0.7,
    standard_scale='var',
    show=True
)

# 手动调整x轴标签（兼容所有版本的终极方法）
plt.gca().set_xticklabels(plt.gca().get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('macrophage_clusters_top10_upregulated_dotplot.png', dpi=300, bbox_inches='tight')



In [169]:
# --- Plot Dot Plot for Cell Type Annotation ---
# Dot plot is an excellent way to show the expression of specific genes in each cluster.
# The size of the dot represents the proportion of cells expressing the gene, and the color intensity represents the average expression level.
# **`use_raw=True`** is a key parameter that ensures we use the normalized and log-transformed raw expression values,
# rather than scaled values, which is more accurate for biological interpretation.
print("Generating dot plot...")
dotplot = sc.pl.dotplot(
    adata_Mac,
    genes_to_plot,
    groupby="leiden_Mac",
    use_raw=True,         # <-- Key! Use data from .raw
    #dendrogram=True,      # Perform hierarchical clustering on clusters, similar clusters will be grouped together
    standard_scale='var', # Normalize by gene to make genes with different expression levels comparable
    cmap='Reds',         # Use blue color scheme
    figsize=(18, 10),     # Set figure size
    return_fig=True,      # Return figure object for further modification
    show=False            # Do not display the figure immediately
)

# --- Customize and Beautify Dot Plot ---
print("Beautifying dot plot...")
dotplot.make_figure() # Generate the figure
main_ax = dotplot.ax_dict['mainplot_ax'] # Get the axes object of the main plot

# Add grid lines to make the figure easier to read
# Add horizontal lines (between genes)
main_ax.hlines(y=np.arange(len(genes_to_plot)) - 0.5, xmin=-0.5, xmax=len(adata_Mac.obs['leiden_Mac'].cat.categories) - 0.5, color='lightgray', lw=0.5, ls='--')
# Add vertical lines (between clusters)
main_ax.vlines(x=np.arange(len(adata_Mac.obs['leiden_Mac'].cat.categories)) - 0.5, ymin=-0.5, ymax=len(genes_to_plot) - 0.5, color='lightgray', lw=0.5, ls='--')

plt.show()


print("\n--- Exporting Raw Data of Dot Plot ---")
mean_expression_df = dotplot.dot_color_df
fraction_df = dotplot.dot_size_df

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print("\n--- Mean Expression Matrix (Color) ---")
#     #print(mean_expression_df)
#     print("\n--- Cell Expression Fraction Matrix (Dot Size) ---")
#     #print(fraction_df)



# 2. Call the function to generate a summary table
final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)

# 3. Convert the resulting DataFrame to a tab-separated string and print it
tsv_output = final_summary_df.to_csv(sep='\t')

#print("--- Gene Expression Level Summary Table (Tab-Separated) ---")
print(tsv_output)

In [170]:
summaries = anno_self.generate_cluster_summaries(final_summary_df)

# 使用 option_context 临时修改设置，不影响后续代码
with pd.option_context('display.max_colwidth', None):
    print(summaries)

In [177]:
macrophage_annotation_simple = {
    "0": "Tissue-Resident Macrophages",
    "1": "Inflammatory Activated Macrophages",
    "2": "Metabolically Active Resting Macrophages",  # 仅保留核心特征
    "3": "Phagocytic Inflammatory Macrophages",
    "4": "Complement-Activated Macrophages",
    "5": "IFN-γ-Responsive Macrophages"
}

In [178]:
# 假设你的分群列是'leiden_Myeloid'，添加注释列
adata_Mac.obs['cell_subtype'] = adata_Mac.obs['leiden_Myeloid'].map(macrophage_annotation_simple)

# 可视化验证
sc.pl.umap(adata_Mac, color='cell_subtype', title='Macrophage Subtype Annotation')

In [179]:
import scanpy as sc
import pandas as pd

# ========================
# 1. 提取adata_Mac的细胞ID-亚型映射（核心步骤）
# ========================
# 确保cell_subtype列存在
if 'cell_subtype' not in adata_Mac.obs.columns:
    raise ValueError("adata_Mac中未找到cell_subtype列，请先确认注释已添加")

# 构建细胞ID到subtype的字典（index为细胞ID）
mac_cell_subtype_map = adata_Mac.obs['cell_subtype'].to_dict()
print(f"共提取 {len(mac_cell_subtype_map)} 个巨噬细胞的亚型注释")

# ========================
# 2. 同步注释到adata_Myeloid（保留原有非巨噬细胞注释）
# ========================
# 初始化：若adata_Myeloid无cell_subtype列，先创建空列
if 'cell_subtype' not in adata_Myeloid.obs.columns:
    adata_Myeloid.obs['cell_subtype'] = pd.NA

# 按细胞ID更新（仅更新巨噬细胞对应的细胞）
# 方法1：批量更新（高效，推荐）
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs.index.map(
    lambda x: mac_cell_subtype_map.get(x, adata_Myeloid.obs.loc[x, 'cell_subtype'])
)

# 方法2：循环更新（新手友好，易调试）
# for cell_id in mac_cell_subtype_map.keys():
#     if cell_id in adata_Myeloid.obs.index:
#         adata_Myeloid.obs.loc[cell_id, 'cell_subtype'] = mac_cell_subtype_map[cell_id]

# ========================
# 3. 验证更新结果
# ========================
# 统计adata_Myeloid中更新后的注释分布
subtype_counts = adata_Myeloid.obs['cell_subtype'].value_counts(dropna=False)
print("\nadata_Myeloid中cell_subtype注释分布：")
print(subtype_counts)

# 验证巨噬细胞注释是否同步成功
# 提取adata_Mac的细胞ID交集
common_cells = set(adata_Mac.obs.index) & set(adata_Myeloid.obs.index)
print(f"\nadata_Myeloid中包含 {len(common_cells)} 个巨噬细胞，注释同步完成")

# 随机抽查5个细胞验证
sample_cells = list(common_cells)[:5]
print("\n随机抽查5个细胞的注释结果：")
for cell in sample_cells:
    print(f"细胞ID: {cell} → 注释: {adata_Myeloid.obs.loc[cell, 'cell_subtype']}")

In [181]:
sc.pl.umap(adata_Myeloid, color='cell_subtype', title='Macrophage Subtype Annotation',frameon=False)

In [1]:
import scanpy as sc
import matplotlib.pyplot as plt

# ========================
# 1. 定义 marker 字典（保持你想要的顺序）
# ========================
# 注意：这里的 Key 必须和 adata_Myeloid.obs['cell_subtype'] 里的名称【完全一致】
# 如果你的 adata 里的名称是全称（如 "Tissue-Resident Macrophages"），请修改下面的 Key
subtype_markers = {
    "Tissue-Resident Macrophages": ['C1QC','C1QA',"LYVE1", "FOLR2", "APOE", "CD163", "C1QA"],
    "Inflammatory Activated Macrophages": ["S100A4", "S100A6", "LGALS1", "VIM", "LYZ"],
    "Metabolically Active Resting Macrophages": ["ATP5E", "ATP5L", "ATP5G2", "GPX1"],
    "Phagocytic Inflammatory Macrophages": ["S100A9", "CTSS", "CTSD", "CD44", "CD36"],
    "Complement-Activated Macrophages": ["C3", "F13A1", "FSCN1"],
    "IFN-γ-Responsive Macrophages": ["GBP1", "ISG15", "IFITM1", "PSMB9"],
    "Classical Monocytes": ["CD14", "S100A8", "CD36"],
    "Non-classical Monocytes": ["CX3CR1", "SELL", "CCR2"],
    "Neutrophils":  ["CSF3R", "NAMPT", "PROK2", "G0S2", "FCGR3B","CXCR2",'CEACAM8'],
    "cDC2": ["CD1C", "CLEC10A", "FCER1A", "HLA-DRA"],
    "ASDC": ["AXL", "SIGLEC6", "PPP1R14A", "HAMP", "DAB2"]

}

# ========================
# 2. 关键步骤：统一 Y 轴顺序
# ========================
# 获取字典里的顺序作为目标顺序
target_order = list(subtype_markers.keys())

# 检查这些名称是否都在数据中
existing_categories = adata_Myeloid.obs['cell_subtype'].cat.categories
# 只保留数据中确实存在的类别，防止报错
final_order = [c for c in target_order if c in existing_categories]

# 【核心代码】重新排列 adata 中的类别顺序
adata_Myeloid.obs['cell_subtype'] = adata_Myeloid.obs['cell_subtype'].cat.reorder_categories(final_order)

# ========================
# 3. 过滤不存在的基因（防错）
# ========================
valid_markers = {}
for subtype, genes in subtype_markers.items():
    existing_genes = [g for g in genes if g in adata_Myeloid.var_names]
    if existing_genes:
        valid_markers[subtype] = existing_genes

# ========================
# 4. 绘制 Dotplot
# ========================
sc.pl.dotplot(
    adata_Myeloid,
    var_names=valid_markers,
    groupby='cell_subtype',
    standard_scale='var',
    figsize=(20, 8),
    # 调整点的大小，让阶梯感更强
    dot_max=0.8,
    smallest_dot=2,
    show=False
)

# 微调边距，确保顶部标签不被遮挡
plt.subplots_adjust(top=0.8) 

plt.savefig('myeloid_staircase_dotplot.png', dpi=300, bbox_inches='tight')
plt.show()

In [192]:
adata_Myeloid.write_h5ad("adata_Myeloid_final.h5ad")

In [5]:
adata_Myeloid = sc.read_h5ad('adata_Myeloid_final.h5ad')

In [6]:
sc.pl.umap(adata_Myeloid, color='cell_subtype', title='Macrophage Subtype Annotation',frameon=False)

In [10]:
sc.pl.umap(adata_Myeloid, color=['dataset','disease']
           ,frameon=False)

In [14]:
sc.pl.umap(adata_Myeloid, color=['C1QA','C1QC','leiden_Myeloid'], frameon=False)

In [13]:
adata_Myeloid

In [16]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. 提取7簇细胞
cluster7_cells = adata_Myeloid[adata_Myeloid.obs['leiden_Myeloid'] == '7']  # 替换为你的分群列名（如cluster）

# 2. 绘制核心标记的表达热图（验证巨噬身份）
sc.pl.dotplot(
    cluster7_cells,
    var_names=['CSF1R', 'MERTK', 'CD68', 'C1QA', 'C1QC', 'CD1C', 'S100A8', 'CD3D'],
    groupby='leiden',  # 若已细分，可改为亚聚类列
    figsize=(10, 4),
    title='Cluster 7 Marker Expression (Including "Corner Cells")'
)

# 3. 可视化UMAP上的C1QA/C1QC表达（定位角落细胞）
sc.pl.umap(
    adata_Myeloid,
    color=['C1QA', 'C1QC'],
    groups=['7'],  # 只显示7簇
    vmin=0, vmax='p95',  # 统一颜色标尺
    title=['C1QA Expression in Cluster 7', 'C1QC Expression in Cluster 7']
)

In [24]:
import scanpy as sc
import numpy as np

# 步骤1：提取簇7的布尔索引（pandas Series）
cluster7_mask = adata_Myeloid.obs['leiden_Myeloid'] == '7'

# 步骤2：提取C1QA/C1QC的表达量（转换为密集数组，解决稀疏矩阵问题）
c1qa_expr = adata_Myeloid[:, 'C1QA'].X.toarray()[:, 0]
c1qc_expr = adata_Myeloid[:, 'C1QC'].X.toarray()[:, 0]

# 步骤3：生成C1QA/C1QC低表达的布尔索引（numpy数组）
low_c1qa_mask = c1qa_expr < 0.1
low_c1qc_mask = c1qc_expr < 0.1

# 步骤4：组合所有条件（统一为numpy布尔数组）
corner_mask = cluster7_mask.values & low_c1qa_mask & low_c1qc_mask

# 步骤5：提取簇7的角落细胞
corner_cells = adata_Myeloid[corner_mask, :].copy()

# 验证：打印角落细胞数量
print(f"簇7中C1QA/C1QC低表达的角落细胞数量：{corner_cells.n_obs}")

# 步骤6：绘制核心标记的小提琴图（修正keys参数）
sc.pl.violin(
    corner_cells,
    keys=['CSF1R', 'CD68', 'MERTK', 'S100A8'],  # 核心修正：var_names → keys       
    rotation=45,
    ylabel='Gene Expression Level'  # 可选：添加y轴标签，增强可读性
)